[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# A Real Client


## What you will be able to do

Write a client for one API as a module of its own: methods that return checked models, exceptions a
program catches by name, and the retries and pages handled inside. Hand the client its session and
its way to wait, and write tests with a fake session that show it works with no server running.


## The idea

### The problem

Every notebook in this guide wrote its requests out in full, in the cell that sent them: the address,
the timeout, the check of the status code, the reading of the body, and lately the retries. Seeing
the pieces was the lesson. A program needs them in many places, and copies drift apart: one call has
a timeout and another has none, one retry loop sends a `POST` again without a key, and a new address
for the API means finding every string that holds the old one.

The code that wants a station also has to know HTTP, down to telling a `404` for a missing station
from a `404` for a mistyped address. And none of it can be checked without the server running, or
without sitting through every wait between retries.

### What a client is

> An **API client** is a module that holds all of a program's code for talking to one API. Its
> methods are named for what the program wants, such as a station, and return objects checked
> against a model. Inside, it sends every request with a timeout, sends again what is worth sending
> again, and turns an error response into an exception named for what went wrong. A client is
> **testable** when what it uses to reach the API, the session that sends requests and the function
> that waits, is handed to it, so that a test can hand it a **fake**: an object with the same method,
> which answers from a list prepared in advance. A **test** is a function that sets up one situation,
> runs the code, and checks what happened with `assert`.

### Why it works that way

- **One place to change.** The address, the timeouts, the retry rules and the reading of errors are
  written once, in the module, so when the API moves, one argument changes.
- **A program catches exceptions, not status codes.** `except NotFoundError` says what is handled.
  The client reads the body to tell the two `404`s apart, so a mistyped address cannot be skipped as
  a missing station.
- **An answer is checked where it arrives.** A response that no longer fits its model raises in the
  client, naming the field that changed, not three functions later as a `KeyError`.
- **The client is handed its parts.** A client that makes its own session can only talk to a server,
  and one that is handed a session can be handed a fake, as the **Composition over Inheritance**
  notebook said of any class that holds a part.
- **A fake can send what a server will not.** The practice API never sends a latitude as text, or a
  `503` on request, and a fake sends either, so a test can check how the client meets a failure it
  may see once a year.
- **Tests are kept, and run again.** A test takes a fraction of a second, so the suite can run after
  every change, and a change that breaks the client is found by a test, not by the program that uses
  it.

### Where you will meet this

A client for a public API is often published as a library, called an SDK. Anthropic's Python SDK is
one. `client.messages.create(...)` returns a Pydantic model, and a failure raises a subclass of
`APIError` named for what went wrong, such as `NotFoundError` for a `404` and `RateLimitError` for
a `429`. It retries connection errors, `408`, `409`, `429` and `5xx` responses 2 times by default,
with a short exponential backoff, and `client.get` and `client.post` reach a path it has no method
for, with the same retries. Stripe's Python library can retry a request after a connection error or
a timeout, and adds an idempotency key to a request that has none, so that its retries are safe.
Later, the **Testing and Packaging** guide takes these tests further: its **Your First Test**
notebook runs functions like them with pytest, and its **Project Layout** notebook gives a module
like this one its place in a project.

### What this notebook covers

- A module written from a cell with `%%writefile`, and imported
- Methods that return checked Pydantic models
- Exceptions of the client's own, and the two `404`s told apart
- Retries in one place, with one idempotency key for all the attempts at a `POST`
- Every page of readings in one loop
- A session and a way to wait, handed to the client
- A fake session, which answers with no server
- A test: a function that checks one behavior with `assert`
- A suite of tests that shows the client works
- Five errors, from readings never asked for to a test that could not fail

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from network_client import NetworkClient, NotFoundError

client = NetworkClient("http://127.0.0.1:8765")
print(client.station("tromso"))

try:
    client.station("narvik")
except NotFoundError as error:
    print("NotFoundError:", error)
```

```
id='tromso' name='Tromso' latitude=69.65 longitude=18.96
NotFoundError: no station with id 'narvik'
```

The program asks for a station, and gets one or an exception that says there is none. The only
address in it is the one the client was made with, and there is no status code or JSON anywhere.


## Setup

Nine imports, the last of them the practice API.

- `requests` builds the responses a fake session answers with, and the connection errors it raises
- `json` writes the body of a response built by hand
- `itertools` takes the first few readings from a loop, with `islice`
- `time` waits, and measures how long a test takes
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here, and removes the module this notebook
  writes, at the end
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads a module, so running a cell again uses the code in its file now
- `practice_api` is the server this guide talks to, started in the background by `start()`

`network_client`, the module this notebook is about, does not exist yet. A cell in Worked examples
writes it, and the cell after that imports it.


In [1]:
import importlib
import itertools
import json
import sys
import time
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### The client, in a module of its own

`%%writefile network_client.py`, as the first line of a cell, is an instruction to Jupyter and
Colab rather than Python: it saves the rest of the cell, without running it, as a file named
`network_client.py` beside the notebook. That file is the client. It holds four models for the
answers it returns, five exceptions for what can go wrong, and `NetworkClient`, in which one method,
`_send`, sends every request. Methods whose names start with an underscore are for the client's own
use, and a program calls the others. Read the file through once. The sections after this one run it
a part at a time:


In [2]:
%%writefile network_client.py
"""A client for the practice API's station network.

The rest of a program imports this module, and never builds a URL, reads a status code or retries a
request. What it needs to know about the API is here: the address, how long to wait, what to send
again, what an error means, and what an answer holds.
"""

import random
import time
import uuid
from datetime import datetime

import requests
from pydantic import BaseModel, ValidationError


class Station(BaseModel):
    id: str
    name: str
    latitude: float
    longitude: float


class Reading(BaseModel):
    station: str
    time: datetime
    temperature_c: float


class ReadingsPage(BaseModel):
    readings: list[Reading]


class Plan(BaseModel):
    id: int
    name: str
    latitude: float
    longitude: float
    elevation_m: float | None = None


class APIError(Exception):
    """Anything that went wrong between this client and the API."""


class NotFoundError(APIError):
    """The API has no station or plan with that id."""


class InvalidRequestError(APIError):
    """The API refused a request as it was sent, with the problems it listed."""

    def __init__(self, message, problems=()):
        super().__init__(message)
        self.problems = list(problems)


class UnavailableError(APIError):
    """Every attempt failed: no response, a 429 or a 5xx."""


class UnexpectedResponseError(APIError):
    """The API answered with something this client was not written to read."""


class NetworkClient:
    """The station network's API as methods, which return checked models or raise an APIError."""

    RETRY_STATUSES = {429, 500, 502, 503, 504}

    def __init__(self, base_url, session=None, attempts=3, timeout=(3.05, 10), sleep=time.sleep,
                 seed=None):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session() if session is None else session
        self.attempts = attempts
        self.timeout = timeout
        self.sleep = sleep
        self.rng = random.Random(seed)

    def station(self, station_id):
        """One station."""
        return self._parse(Station, self._send("GET", f"/stations/{station_id}"))

    def readings(self, station, per_page=100):
        """Every reading from a station. Each page is fetched when the loop reaches it."""
        response = self._send("GET", "/network/readings", params={"station": station, "per_page": per_page})
        while True:
            for reading in self._parse(ReadingsPage, response).readings:
                yield reading
            if "next" not in response.links:
                return
            response = self._send("GET", response.links["next"]["url"])

    def create_plan(self, name, latitude, longitude, **fields):
        """A new plan, created once however many attempts it takes."""
        plan = {"name": name, "latitude": latitude, "longitude": longitude, **fields}
        return self._parse(Plan, self._send("POST", "/network/plans", json=plan))

    def get(self, path, **params):
        """The JSON at a path this client has no method for, with the same retries and errors."""
        response = self._send("GET", path, params=params)
        try:
            return response.json()
        except requests.JSONDecodeError as error:
            raise UnexpectedResponseError(f"GET {path} did not answer with JSON") from error

    def _send(self, method, path, **kwargs):
        """The response to a request, from the first attempt that gets one worth keeping."""
        url = path if path.startswith("http") else self.base_url + path    # a Link header's address is whole
        headers = {"X-Request-Id": uuid.uuid4().hex}                       # one id for every attempt
        if method == "POST":
            headers["Idempotency-Key"] = str(uuid.uuid4())                 # so a retry cannot create twice
        for attempt in range(1, self.attempts + 1):
            response = None
            try:
                response = self.session.request(method, url, headers=headers, timeout=self.timeout, **kwargs)
            except (requests.ConnectionError, requests.Timeout) as error:
                outcome = type(error).__name__
            else:
                if response.status_code not in self.RETRY_STATUSES:
                    self._raise_for(response)
                    return response
                outcome = response.status_code
            if attempt == self.attempts:
                raise UnavailableError(f"{method} {path}: {outcome} on attempt {attempt} of {self.attempts}")
            self.sleep(self._wait(attempt, response))

    def _wait(self, attempt, response):
        """Seconds before the next attempt: what Retry-After asks for, or a backoff with jitter."""
        retry_after = "" if response is None else response.headers.get("Retry-After", "")
        if retry_after.isdigit():
            return float(retry_after)
        return self.rng.uniform(0, 2 ** (attempt - 1))

    def _raise_for(self, response):
        """Raise the APIError that says what went wrong, if anything did."""
        if response.status_code < 400:
            return
        body = response.json() if "json" in response.headers.get("Content-Type", "") else {}
        message = body.get("error", f"{response.status_code} {response.reason}")
        if response.status_code == 404 and not message.startswith("nothing at"):
            raise NotFoundError(message)
        if response.status_code in (400, 422):
            raise InvalidRequestError(message, body.get("problems", []))
        raise APIError(f"{response.status_code}: {message}")

    def _parse(self, model, response):
        """The body as the model, checked strictly: an UnexpectedResponseError if it does not fit."""
        try:
            return model.model_validate_json(response.content, strict=True)
        except ValidationError as error:
            found = [f"{'.'.join(map(str, problem['loc'])) or 'the body'}: {problem['msg']}"
                     for problem in error.errors()]
            raise UnexpectedResponseError(f"{model.__name__}: {'; '.join(found)}") from error


Writing network_client.py


The cell printed `Writing network_client.py`, and prints `Overwriting` whenever it runs again.
Importing the file runs it, as Setup's `import practice_api` runs `practice_api.py`. `reload` runs it
again if an earlier run of this cell imported it, and the `from` line comes after `reload`, so its
names are the ones in the file now: the **Modules and Imports** notebook showed that names taken by a
`from` line before a `reload` keep pointing at the old code.


In [3]:
import network_client
importlib.reload(network_client)    # runs the file as it is now, not a copy imported earlier
from network_client import (APIError, InvalidRequestError, NetworkClient, NotFoundError,
                            UnavailableError, UnexpectedResponseError)

client = NetworkClient(BASE)
tromso = client.station("tromso")
print(tromso)
print(type(tromso).__name__, "| latitude:", tromso.latitude, type(tromso.latitude).__name__)


id='tromso' name='Tromso' latitude=69.65 longitude=18.96
Station | latitude: 69.65 float


`client.station("tromso")` sent `GET /stations/tromso`, checked the body against `Station`, and
returned a `Station`, whose fields are attributes with the types the model declares. `_parse` checks
with `model_validate_json` and `strict=True`, the checking the **Schemas and Validation** notebook
ended on, which converts nothing: a latitude sent as text would raise, and a later section fakes an
answer that does.

### Errors named for what went wrong

`_raise_for` turns an error response into one of the module's exceptions, and reads the body to
choose which. Four calls, which fail in four ways:


In [4]:
mistyped = NetworkClient(f"{BASE}/api")          # an address with a path this API does not have

for what, call in [("a missing station", lambda: client.station("narvik")),
                   ("a mistyped address", lambda: mistyped.station("tromso")),
                   ("a plan with a problem", lambda: client.create_plan("Lakselv", 170.05, 24.97)),
                   ("a page that is not JSON", lambda: client.get("/"))]:
    try:
        call()
    except APIError as error:
        print(f"{what}: {type(error).__name__}: {error}")
        if isinstance(error, InvalidRequestError):
            print("   ", error.problems)


a missing station: NotFoundError: no station with id 'narvik'
a mistyped address: APIError: 404: nothing at /api/stations/tromso
a plan with a problem: InvalidRequestError: the plan has problems
    [{'field': 'latitude', 'problem': 'must be a number from -90 to 90'}]
a page that is not JSON: UnexpectedResponseError: GET / did not answer with JSON


The first two calls both got `404`. One body said there is no station with that id, so the client
raised `NotFoundError`. The other said there is nothing at that address, which is a mistake in how
the client was made, so the client raised the plain `APIError`, which `except NotFoundError` does
not catch. `InvalidRequestError` keeps the problems the `422` listed, in an attribute a program can
read, and the HTML page at `/` became an `UnexpectedResponseError` rather than requests'
`JSONDecodeError`. `except APIError` caught all four, because every exception in the module is a
subclass of it, like the family of exceptions the **Exceptions as Classes** notebook built. A
program catches only the ones it can handle:


In [5]:
for station_id in ["tromso", "narvik", "oslo"]:
    try:
        station = client.station(station_id)
    except NotFoundError:
        print(f"{station_id}: no such station, skipped")
        continue
    print(f"{station.name}: {station.latitude}, {station.longitude}")


Tromso: 69.65, 18.96
narvik: no such station, skipped
Oslo: 59.91, 10.75


The loop skips Narvik and nothing else. Given `mistyped` instead of `client`, it stops at Tromso with
an `APIError`, which is right: every station would fail the same way, and a loop that skipped them
all would report a network with no stations, the mistake the **Status Codes** notebook warned of.

### Retries, and the waits between them

`_send` makes up to three attempts. It sends a request again after a lost connection, a timeout, a
`429` or a `5xx`, waits for the seconds a `Retry-After` header names, or else for a backoff with
jitter as the **Errors and Retries** notebook did, and sends the same `X-Request-Id` with every
attempt. To wait, it calls the `sleep` it was given, and `pause`, below, waits and notes how long.
`/network/unstable` fails twice before it answers, and six requests in a row to `/network/latest` go
past its limit of 5 every 2 seconds:


In [6]:
waits = []


def pause(seconds):
    """Wait, and note how long."""
    waits.append(round(seconds, 2))
    time.sleep(seconds)


client = NetworkClient(BASE, sleep=pause, seed=7)          # seeded, so the jitter is the same on every run
print("unstable: attempt", client.get("/network/unstable")["attempt"], "| waits:", waits)

waits.clear()
for _ in range(6):
    latest = client.get("/network/latest")
print("latest, 6 times:", len(latest), "stations | waits:", waits)


unstable: attempt 3 | waits: [0.32, 1.0]
latest, 6 times: 3 stations | waits: [2.0]


The first attempt at `/network/unstable` lost its connection, and the client waited a random 0.32
seconds. The second got `503` with `Retry-After: 1` and waited exactly that, and the third was
answered. The sixth request to `/network/latest` got `429` with `Retry-After: 2`, waited, and was
answered, so the loop never met the limit. When every attempt fails, `_send` raises
`UnavailableError`, naming the last failure:


In [7]:
waits.clear()
try:
    client.get("/hang-up")
except UnavailableError as error:
    print("UnavailableError:", error, "| waits:", waits)


UnavailableError: GET /hang-up: ConnectionError on attempt 3 of 3 | waits: [0.15, 1.3]


Two waits for three attempts, since nothing follows the last. A `POST` also carries an
`Idempotency-Key`, one for all its attempts, so the lost response in the **Sending Data** notebook
cannot create a plan twice through this client, and `create_plan` needs nothing more to be safe to
send again.

### Every page, in one loop

`readings` contains `yield`, as the `__iter__` methods in the **Context Managers and Iterators**
notebook did, so calling it returns a generator, which runs the method a reading at a time as a loop
asks. It hands out one page's readings, then follows the `Link` header's `next` address to the next
page, as the **Pagination** notebook's loops did, until there is none. The practice API's access log,
which the **Authentication** notebook read, shows what one loop sent:


In [8]:
logged = len(practice_api.access_log())
readings = list(client.readings("oslo", per_page=30))

print(len(readings), "readings, from these requests:")
for line in practice_api.access_log()[logged:]:
    print("  ", line.split('"')[1])                       # the request, which the log line quotes
print("first:", readings[0].time, readings[0].temperature_c, "| last:", readings[-1].time, readings[-1].temperature_c)


72 readings, from these requests:
   GET /network/readings?station=oslo&per_page=30 HTTP/1.1
   GET /network/readings?station=oslo&per_page=30&page=2 HTTP/1.1
   GET /network/readings?station=oslo&per_page=30&page=3 HTTP/1.1
first: 2026-02-26 10:00:00+00:00 -3.4 | last: 2026-03-01 09:00:00+00:00 -4.2


Three pages of 30 made one list of 72 readings, and `time` arrived as a `datetime`, the type
`Reading` declares. A page is fetched only when the loop reaches it, so a loop that stops early
sends less. `itertools.islice` takes the first five readings and stops asking:


In [9]:
logged = len(practice_api.access_log())
first_five = list(itertools.islice(client.readings("oslo", per_page=30), 5))
print(len(first_five), "readings, from", len(practice_api.access_log()) - logged, "request")


5 readings, from 1 request


One request, for the first page, and the other two were never sent. Common errors shows the other
side of this: a generator sends nothing at all until a loop asks it for a reading.

### What the client is given: a session, and a way to wait

`NetworkClient` does not decide how its requests are sent or how it waits. It is given a session,
making a `requests.Session` only when it is given none, and `sleep`, which is `time.sleep` unless
something else is passed. A session keeps what every request should carry, and
reuses one connection for requests to the same host, as requests' documentation notes. GitHub's API
refuses a request with no `User-Agent` header, and asks for one that names the user or the
application, so that GitHub can contact whoever sent it. A program sets that once, on the session it
gives the client. And `waits.append`, given as `sleep`, notes every wait and waits for none:


In [10]:
session = requests.Session()
session.headers["User-Agent"] = "station-report/1.0"       # every request names the program that sent it

waits = []
client = NetworkClient(BASE, session=session, sleep=waits.append, seed=7)     # seeded, as before

started = time.monotonic()
print("unstable: attempt", client.get("/network/unstable")["attempt"], "| waits noted:", [round(wait, 2) for wait in waits])
print("seconds spent:", round(time.monotonic() - started))
print("User-Agent, as the server received it:", client.get("/echo/headers")["headers"]["User-Agent"])


unstable: attempt 3 | waits noted: [0.32, 1.0]
seconds spent: 0
User-Agent, as the server received it: station-report/1.0


The same waits as before, noted and not waited: the practice API does not check that a client waited,
so the third attempt was answered at once. Nothing in `NetworkClient` uses `time.sleep` or
`requests.Session` except as a default, so the code that makes a client decides what they are: the
real ones in a program, and stand-ins in a test.

### A fake session: answers with no server

The client calls one method of its session, `request`, with a method, an address and keyword
arguments, so any object with that method can stand in for a session. `FakeSession` answers each
request with the next of the answers it was given, raises an answer that is an exception, as a real
session raises for a lost connection, and keeps every request, so a test can check what was sent.
`answer` builds a `requests.Response` by hand. The requests library has no public way to give a
response its body, so `answer` sets `_content`, where a `Response` keeps it. A library such as
`responses`, made for testing code that uses requests, builds responses for you. `api.test` is a
made-up address, since `.test` is a name reserved for testing, and the fake never tries to reach it:


In [11]:
class FakeSession:
    """Stands in for requests.Session: answers with the answers it was given, in order, and keeps every request."""

    def __init__(self, *answers):
        self.answers = list(answers)
        self.requests = []

    def request(self, method, url, **kwargs):
        self.requests.append((method, url, kwargs))
        answer = self.answers.pop(0)
        if isinstance(answer, Exception):
            raise answer
        return answer


def answer(status, body, headers=None):
    """A response with a status code, a JSON body and any other headers, built with no server."""
    response = requests.Response()
    response.status_code = status
    response.headers.update({"Content-Type": "application/json", **(headers or {})})
    response._content = json.dumps(body).encode()          # where a Response keeps its body
    return response


TROMSO = {"id": "tromso", "name": "Tromso", "latitude": 69.65, "longitude": 18.96}

fake = FakeSession(requests.ConnectionError("the connection closed"),
                   answer(503, {"error": "the service is busy: try again"}, {"Retry-After": "1"}),
                   answer(200, TROMSO))
waits = []
client = NetworkClient("http://api.test", session=fake, sleep=waits.append, seed=7)     # seeded, as before

print(client.station("tromso"), "| waits noted:", [round(wait, 2) for wait in waits])
for method, url, kwargs in fake.requests:
    print(" ", method, url, "| timeout:", kwargs["timeout"])
print("  X-Request-Ids among them:", len({kwargs["headers"]["X-Request-Id"] for method, url, kwargs in fake.requests}))


id='tromso' name='Tromso' latitude=69.65 longitude=18.96 | waits noted: [0.32, 1.0]
  GET http://api.test/stations/tromso | timeout: (3.05, 10)
  GET http://api.test/stations/tromso | timeout: (3.05, 10)
  GET http://api.test/stations/tromso | timeout: (3.05, 10)
  X-Request-Ids among them: 1


The station came back after a lost connection and a `503`, with the same waits noted, and no server
was involved. The fake's record shows three attempts at one address, all with the client's timeout
and one `X-Request-Id` between them. A fake can also send what the practice API never does, such as a
latitude sent as text:


In [12]:
fake = FakeSession(answer(200, {**TROMSO, "latitude": "69.65"}))
try:
    NetworkClient("http://api.test", session=fake).station("tromso")
except UnexpectedResponseError as error:
    print("UnexpectedResponseError:", error)


UnexpectedResponseError: Station: latitude: Input should be a valid number


`_parse` named the model and the field. Without `strict=True`, Pydantic would have turned the text
into a number, and the client would have gone on trusting an API whose answers had changed.

### A test: one behavior, checked with assert

A test is a function. It sets up one situation, runs the code, and checks the result with `assert`,
which does nothing when its condition is true and raises `AssertionError` when it is false. A test
that raises nothing has passed. Its name says what it checks, and starts with `test_`, which is how
pytest, in the **Your First Test** notebook of the **Testing and Packaging** guide, finds the
functions to run. A test that expects an exception needs `else`, which runs when the `try` raised
nothing, because a test that only catches the exception also passes when none comes. `run_tests`
runs tests and reports on every one, as pytest does in more detail:


In [13]:
def test_a_missing_station_raises_not_found_error():
    fake = FakeSession(answer(404, {"error": "no station with id 'narvik'"}))
    client = NetworkClient("http://api.test", session=fake)
    try:
        client.station("narvik")
    except NotFoundError as error:
        assert "narvik" in str(error)
    else:
        raise AssertionError("station('narvik') raised nothing")


def run_tests(*tests):
    """Run tests, and report every one that fails, not only the first."""
    failed = 0
    for test in tests:
        try:
            test()
        except Exception as error:
            failed += 1
            print(f"FAILED {test.__name__}: {type(error).__name__}: {error}")
        else:
            print(f"passed {test.__name__}")
    print(f"{len(tests) - failed} passed, {failed} failed")


run_tests(test_a_missing_station_raises_not_found_error)


passed test_a_missing_station_raises_not_found_error
1 passed, 0 failed


`run_tests` runs every test inside a `try`, so one that fails is reported and the rest still run. The
**Testing Failure** notebook replaces a test's `try`, `except` and `else` with `pytest.raises`, and
Common errors shows what goes wrong when the `else` is left out.

### Tests that show the client works

The pieces of this notebook, in one suite. Every test gives `NetworkClient` a `FakeSession`, and a
stand-in for `sleep` wherever the client would wait, and checks one behavior from a section above.
None of them needs a server:


In [14]:
def test_a_station_is_checked_and_returned():
    fake = FakeSession(answer(200, TROMSO))
    station = NetworkClient("http://api.test", session=fake).station("tromso")
    assert station.latitude == 69.65
    assert fake.requests[0][:2] == ("GET", "http://api.test/stations/tromso")


def test_a_mistyped_address_is_not_a_missing_station():
    fake = FakeSession(answer(404, {"error": "nothing at /api/stations/tromso"}))
    try:
        NetworkClient("http://api.test/api", session=fake).station("tromso")
    except NotFoundError:
        raise AssertionError("a mistyped address was reported as a missing station")
    except APIError as error:
        assert "nothing at" in str(error)
    else:
        raise AssertionError("station() raised nothing")


def test_a_changed_answer_raises_unexpected_response_error():
    fake = FakeSession(answer(200, {**TROMSO, "latitude": "69.65"}))
    try:
        NetworkClient("http://api.test", session=fake).station("tromso")
    except UnexpectedResponseError as error:
        assert "latitude" in str(error)
    else:
        raise AssertionError("a latitude sent as text was accepted")


def test_a_503_is_sent_again_after_its_retry_after():
    waits = []
    fake = FakeSession(answer(503, {"error": "busy"}, {"Retry-After": "2"}), answer(200, TROMSO))
    NetworkClient("http://api.test", session=fake, sleep=waits.append).station("tromso")
    assert waits == [2.0]
    assert len(fake.requests) == 2


def test_the_attempts_run_out():
    waits = []
    fake = FakeSession(*[requests.ConnectionError("the connection closed")] * 3)
    try:
        NetworkClient("http://api.test", session=fake, sleep=waits.append).station("tromso")
    except UnavailableError:
        assert len(fake.requests) == 3 and len(waits) == 2
    else:
        raise AssertionError("three lost connections raised nothing")


def test_every_attempt_at_a_plan_sends_one_idempotency_key():
    plan = {"id": 1, "name": "Alta", "latitude": 69.97, "longitude": 23.27}
    fake = FakeSession(requests.ConnectionError("the connection closed"), answer(201, plan))
    NetworkClient("http://api.test", session=fake, sleep=lambda seconds: None).create_plan("Alta", 69.97, 23.27)
    keys = [kwargs["headers"]["Idempotency-Key"] for method, url, kwargs in fake.requests]
    assert len(keys) == 2 and keys[0] == keys[1]


def test_readings_follow_the_next_link():
    link = '<http://api.test/network/readings?station=oslo&page=2>; rel="next"'
    first = {"station": "oslo", "time": "2026-02-26T10:00Z", "temperature_c": -3.4}
    second = {"station": "oslo", "time": "2026-02-26T11:00Z", "temperature_c": -2.5}
    fake = FakeSession(answer(200, {"readings": [first]}, {"Link": link}), answer(200, {"readings": [second]}))
    readings = list(NetworkClient("http://api.test", session=fake).readings("oslo"))
    assert [reading.temperature_c for reading in readings] == [-3.4, -2.5]
    assert fake.requests[1][1] == "http://api.test/network/readings?station=oslo&page=2"


started = time.monotonic()
run_tests(test_a_missing_station_raises_not_found_error, test_a_station_is_checked_and_returned,
          test_a_mistyped_address_is_not_a_missing_station, test_a_changed_answer_raises_unexpected_response_error,
          test_a_503_is_sent_again_after_its_retry_after, test_the_attempts_run_out,
          test_every_attempt_at_a_plan_sends_one_idempotency_key, test_readings_follow_the_next_link)
print("in less than a second:", time.monotonic() - started < 1)


passed test_a_missing_station_raises_not_found_error
passed test_a_station_is_checked_and_returned
passed test_a_mistyped_address_is_not_a_missing_station
passed test_a_changed_answer_raises_unexpected_response_error
passed test_a_503_is_sent_again_after_its_retry_after
passed test_the_attempts_run_out
passed test_every_attempt_at_a_plan_sends_one_idempotency_key
passed test_readings_follow_the_next_link
8 passed, 0 failed
in less than a second: True


### Where each part came from

| In the tests | What it relies on | The section that showed it |
|---|---|---|
| a `FakeSession` given as `session` | a client that sends every request through the session it is given | What the client is given: a session, and a way to wait |
| `answer(...)`, and an exception among the answers | a response built by hand, and a lost connection raised on purpose | A fake session: answers with no server |
| `waits.append` given as `sleep` | waits noted, and not waited | What the client is given: a session, and a way to wait |
| `except NotFoundError: raise AssertionError(...)` | a body that tells a missing station from a mistyped address | Errors named for what went wrong |
| `"latitude" in str(error)` | every answer checked strictly, with the field that does not fit named | The client, in a module of its own, and the **Schemas and Validation** notebook |
| `waits == [2.0]` | the seconds `Retry-After` names, waited before the next attempt | Retries, and the waits between them |
| one `Idempotency-Key` in both attempts | a `POST` that is safe to send again | the **Sending Data** notebook |
| the second request, to the `next` address | pages followed through the `Link` header | Every page, in one loop, and the **Pagination** notebook |
| `else: raise AssertionError(...)` | a test that fails when nothing is raised | A test: one behavior, checked with assert |

Eight tests passed in less than a second, with no server running. The client they tested is the one
that talked to the practice API above, unchanged: only what it was handed differs. A change to
`network_client.py` that broke a behavior, such as a `_raise_for` that raised `NotFoundError` for
every `404`, would print `FAILED` beside the test that checks it the next time the suite ran. Saved
in a file named `test_network_client.py`, the same functions are a suite that pytest can run.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/14-a-real-client-solutions.ipynb).

**1.** Make a `NetworkClient` for the practice API, get the station `oslo`, and print its name, its
latitude and its longitude.


In [15]:
# your code here


**2.** Ask the client for the station `bodo`, catch the exception, and print its class name and its
message.


In [16]:
# your code here


**3.** Loop over the readings from `bergen`, 25 to a page, and print how many there are and the
lowest temperature among them.


In [17]:
# your code here


**4.** Use `create_plan` for a plan named Alta, at latitude 69.97 and longitude 23.27, and print the
plan. Then try a plan named Kautokeino, at latitude 95 and longitude 23.04, and print each problem
the exception lists.


In [18]:
# your code here


**5.** Make a client with a `FakeSession` that answers a `503` with `Retry-After: 2`, and then the
Tromso station, and with `waits.append` as its `sleep`. Print the station's name and the waits.


In [19]:
# your code here


**6.** Write a test, `test_a_422_lists_its_problems`, that gives the client a fake `422` with a
problem for the `latitude` field, checks that `create_plan` raises `InvalidRequestError` whose
`problems` name that field, and fails if nothing is raised. Run it with `run_tests`.


In [20]:
# your code here


## Common errors

### No error, and no request: readings that nothing has looped over


In [21]:
client = NetworkClient(BASE)
logged = len(practice_api.access_log())
bodo = client.readings("bodo")                  # no station has the id bodo

print(type(bodo).__name__, "| requests sent:", len(practice_api.access_log()) - logged)


generator | requests sent: 0


`readings` contains `yield`, so calling it runs none of its code. It returns a generator, which
starts running only when a loop asks it for the first reading, so nothing was sent and nothing could
fail, and the mistaken id surfaces later, in whatever loop reads `bodo`. Put the `try` around the
loop, where the request is sent:


In [22]:
try:
    for reading in client.readings("bodo"):
        print(reading.temperature_c)
except InvalidRequestError as error:
    print("InvalidRequestError:", error)


InvalidRequestError: no station has the id 'bodo'


### TypeError: QuickFake.request() got an unexpected keyword argument 'headers'


In [23]:
class QuickFake:
    """A fake written for the arguments its author expected."""

    def request(self, method, url):
        return answer(200, TROMSO)


NetworkClient("http://api.test", session=QuickFake()).station("tromso")


TypeError: QuickFake.request() got an unexpected keyword argument 'headers'

A fake has to accept everything the client passes. `_send` passes `headers` and `timeout` as keyword
arguments, and `params` or `json` as well for some requests, and `QuickFake.request` accepts only a
method and an address. Accept any keyword argument with `**kwargs`, as `FakeSession.request` does:


In [24]:
class QuickFake:
    """A fake that accepts whatever the client passes."""

    def request(self, method, url, **kwargs):
        return answer(200, TROMSO)


print(NetworkClient("http://api.test", session=QuickFake()).station("tromso"))


id='tromso' name='Tromso' latitude=69.65 longitude=18.96


### IndexError: pop from empty list


In [25]:
fake = FakeSession(requests.ConnectionError("the connection closed"))       # for a test that the attempts run out
NetworkClient("http://api.test", session=fake, sleep=waits.append).station("tromso")


IndexError: pop from empty list

The client makes three attempts, and the fake had an answer for one, so the second attempt found its
list empty. The `IndexError` comes from the fake rather than the client, so a test that expected
`UnavailableError` would fail with an error about its own setup. Give a fake an answer for every
request the client will send:


In [26]:
fake = FakeSession(*[requests.ConnectionError("the connection closed")] * 3)
try:
    NetworkClient("http://api.test", session=fake, sleep=waits.append).station("tromso")
except UnavailableError as error:
    print("UnavailableError:", error)


UnavailableError: GET /stations/tromso: ConnectionError on attempt 3 of 3


### No error, and a test that passed when nothing was raised: a try without an else


In [27]:
def test_a_missing_station_raises():
    fake = FakeSession(answer(200, TROMSO))          # a station, so station() raises nothing
    try:
        NetworkClient("http://api.test", session=fake).station("narvik")
    except NotFoundError:
        pass


run_tests(test_a_missing_station_raises)


passed test_a_missing_station_raises
1 passed, 0 failed


The test is meant to show that asking for a missing station raises `NotFoundError`. This fake
answers with a station, so nothing is raised, and the test passes anyway: `except` runs only when an
exception comes, and a test that ends without one has passed. A test that cannot fail checks
nothing. Add `else`, and fail there:


In [28]:
def test_a_missing_station_raises():
    fake = FakeSession(answer(200, TROMSO))
    try:
        NetworkClient("http://api.test", session=fake).station("narvik")
    except NotFoundError:
        pass
    else:
        raise AssertionError("station('narvik') raised nothing")


run_tests(test_a_missing_station_raises)


FAILED test_a_missing_station_raises: AssertionError: station('narvik') raised nothing
0 passed, 1 failed


Now the test fails, as it should with this fake. To check that a test can fail, give it a situation
in which the behavior it checks is missing, and see it report `FAILED`.

### No error, and a test that takes seconds: a client given no sleep


In [29]:
def test_two_503s_then_a_station():
    fake = FakeSession(answer(503, {"error": "busy"}, {"Retry-After": "1"}),
                       answer(503, {"error": "busy"}, {"Retry-After": "1"}), answer(200, TROMSO))
    assert NetworkClient("http://api.test", session=fake).station("tromso").name == "Tromso"


started = time.monotonic()
run_tests(test_two_503s_then_a_station)
print("seconds:", round(time.monotonic() - started))


passed test_two_503s_then_a_station
1 passed, 0 failed
seconds: 2


The test passed, and took 2 seconds: it gave the client no `sleep`, so the client used `time.sleep`
and waited out both `Retry-After`s. A suite of tests like it grows slow enough that people stop
running it. Give the client `waits.append`, which also lets the test check the waits:


In [30]:
def test_two_503s_then_a_station():
    waits = []
    fake = FakeSession(answer(503, {"error": "busy"}, {"Retry-After": "1"}),
                       answer(503, {"error": "busy"}, {"Retry-After": "1"}), answer(200, TROMSO))
    assert NetworkClient("http://api.test", session=fake, sleep=waits.append).station("tromso").name == "Tromso"
    assert waits == [1.0, 1.0]


started = time.monotonic()
run_tests(test_two_503s_then_a_station)
print("seconds:", round(time.monotonic() - started))


passed test_two_503s_then_a_station
1 passed, 0 failed
seconds: 0


Last, the notebook is finished with its module, so this cell removes the file it wrote. Run the
`%%writefile` cell again to bring it back:


In [31]:
Path("network_client.py").unlink()

print("network_client.py still there:", Path("network_client.py").exists())


network_client.py still there: False


## Recap

- A client is a module that holds a program's code for one API: methods named for what the program
  wants, which return checked models.
- Exceptions of the client's own, all subclasses of one `APIError`, let a program catch what it can
  handle by name. Reading the body tells a missing station from a mistyped address.
- The retries, the waits, one `X-Request-Id` for all the attempts, and an `Idempotency-Key` on a
  `POST` are written once, in the method that sends every request.
- A method with `yield` follows the pages, fetching one when the loop reaches it, and sends nothing
  until a loop asks it for something.
- A client that is handed its session and its way to wait can be handed a fake session and a
  function that notes the waits.
- A test is a function that checks one behavior with `assert`. It has to fail when the behavior is
  missing, an exception that never came included.


## What is next

The **Your First API Server** notebook. Every notebook in this guide so far has been on the client's
side, sending requests and reading what came back. That notebook changes sides: it builds a server
with FastAPI, a route at a time, and reads the documentation FastAPI writes for it.


---

&#8592; **Previous:** [Sending Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/13-sending-data.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Your First API Server](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/15-your-first-api-server.ipynb) &#8594;
